## Section 4: Configure Training Pipeline

In [4]:
# Build Audio Spectrogram Transformer Model
class PatchEmbedding(nn.Module):
    """Convert spectrogram into patch embeddings"""
    def __init__(self, input_shape, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
        
        self.proj = nn.Conv2d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention module"""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class TransformerBlock(nn.Module):
    """Transformer encoder block"""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class AudioSpectrogramTransformer(nn.Module):
    """Audio Spectrogram Transformer for sound event detection"""
    def __init__(self, config):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            config['input_shape'],
            config['patch_size'],
            config['embed_dim']
        )
        
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config['embed_dim']))
        
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config['embed_dim'],
                config['num_heads'],
                config['mlp_ratio'],
                config['dropout']
            )
            for _ in range(config['num_layers'])
        ])
        
        self.norm = nn.LayerNorm(config['embed_dim'])
        self.head = nn.Linear(config['embed_dim'], config['num_classes'])
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_module_weights)
    
    def _init_module_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.head(x)
        return x

# Build model
model_config = config['model'].copy()
model_config['input_shape'] = [config['preprocessing']['n_mels'], 1000]  # Adjusted for duration
model = AudioSpectrogramTransformer(model_config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model built successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

KeyError: 'patch_size'

## Section 3: Build Audio Spectrogram Transformer Model

This section builds the AST model from `models/ast_model.py` with the configured architecture.

In [3]:
# Dataset Loading and Preparation
print("Loading datasets from data/processed/ directory...")

# List all processed data files
processed_dir = PROJECT_ROOT / "data" / "processed"
train_files = sorted(processed_dir.glob("train/*.npy"))
val_files = sorted(processed_dir.glob("val/*.npy"))
test_files = sorted(processed_dir.glob("test/*.npy"))

print(f"\nDataset Summary:")
print(f"  - Training samples: {len(train_files)}")
print(f"  - Validation samples: {len(val_files)}")
print(f"  - Test samples: {len(test_files)}")
print(f"  - Total samples: {len(train_files) + len(val_files) + len(test_files)}")

# Load a sample to check data shape
if train_files:
    sample_data = np.load(train_files[0])
    print(f"\nSample spectrogram shape: {sample_data.shape}")
    print(f"Data type: {sample_data.dtype}")
    print(f"Data range: [{sample_data.min():.4f}, {sample_data.max():.4f}]")

Loading datasets from data/processed/ directory...

Dataset Summary:
  - Training samples: 0
  - Validation samples: 0
  - Test samples: 0
  - Total samples: 0


## Section 2: Load and Prepare Datasets

In [2]:
# Load Configuration
config_path = PROJECT_ROOT / "configs" / "config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully!")
print(f"Project root: {PROJECT_ROOT}")
print(f"\nModel Configuration:")
print(f"  - Model name: {config['model']['name']}")
print(f"  - Number of classes: {config['model']['num_classes']}")
print(f"  - Embedding dimension: {config['model']['embed_dim']}")
print(f"  - Number of layers: {config['model']['num_layers']}")
print(f"\nTraining Configuration:")
print(f"  - Batch size: {config['training']['batch_size']}")
print(f"  - Learning rate: {config['training']['optimizer']['lr']}")
print(f"  - Maximum epochs: {config['training']['num_epochs']}")

Configuration loaded successfully!
Project root: /Users/dangchauanh/Downloads/agent-artifacts-zip_4a7a084c-cf0c-43c1-88b8-c7669caa2676_1772094907/audio_event_detection

Model Configuration:


KeyError: 'name'

In [1]:
# Import Required Libraries
import os
import sys
import yaml
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F

# Utilities
from tqdm import tqdm
import librosa
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, average_precision_score
)

warnings.filterwarnings('ignore')

# Setup paths
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Setup plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

Using device: cpu


## Section 1: Import Required Libraries and Configuration

# Audio Event Detection - Complete Training & Evaluation Pipeline
## Audio Spectrogram Transformer for Emergency Sound Detection

This notebook provides a comprehensive end-to-end pipeline for:
1. Loading and preparing datasets
2. Building AST model architecture  
3. Configuring training pipeline
4. Training the model with validation
5. Evaluating performance on test set
6. Visualizing results and metrics
7. Running inference on new audio files
8. Exporting models for deployment

**Project Root Structure:**
```
audio-event-detection/
├── data/processed/            # Preprocessed spectrograms
├── models/ast_model.py        # AST architecture
├── utils/                     # Custom modules
├── scripts/                   # Training scripts
├── configs/config.yaml        # Configuration
└── notebooks/                 # Analysis notebooks
```